In [1]:
import re
import string
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer

from cleanlab import Datalab

In [2]:
data = pd.read_csv("../data/full.csv")
data.head()

,Headline,Stance,articleBody
0,Police find mass graves with at least '15 bodi...,unrelated,Danny Boyle is directing the untitled film\r\n...
1,Hundreds of Palestinians flee floods in Gaza a...,agree,Hundreds of Palestinians were evacuated from t...
2,"Christian Bale passes on role of Steve Jobs, a...",unrelated,30-year-old Moscow resident was hospitalized w...
3,HBO and Apple in Talks for $15/Month Apple TV ...,unrelated,(Reuters) - A Canadian soldier was shot at the...
4,Spider burrowed through tourist's stomach and ...,disagree,"Fear not arachnophobes, the story of Bunbury's..."


In [3]:
data.shape

(75385, 3)

In [4]:
raw_texts, raw_headlines, labels = data["articleBody"].values, data["Headline"].values, data["Stance"].values
num_classes = len(set(labels))

print(f"This dataset has {num_classes} classes.")
print(f"Classes: {set(labels)}")

This dataset has 4 classes.
Classes: {'discuss', 'agree', 'disagree', 'unrelated'}


In [5]:
# create a new df with the same features but keep the number of Stances the same, meaning that if there are only 1000 examples of "agree", then we will only keep 1,000 examples of "disagree", "discuss", and "unrelated"
df_balanced = data.groupby("Stance").apply(lambda x: x.sample(n=1000, random_state=42)).reset_index(drop=True)
raw_texts, raw_headlines, labels = df_balanced["articleBody"].values, df_balanced["Headline"].values, df_balanced["Stance"].values
num_classes = len(set(labels))

print(f"This dataset has {num_classes} classes.")
print(f"Classes: {set(labels)}")


This dataset has 4 classes.
Classes: {'discuss', 'agree', 'disagree', 'unrelated'}


In [6]:
i = 1  # change this to view other examples from the dataset
print(f"Example Label: {labels[i]}")
print(f"Example Text: {raw_texts[i]}")
print(f"\n\nExample Headline: {raw_headlines[i]}")

Example Label: agree
Example Text: The woman who claimed she had a third breast was lying and the story is a hoax.


Example Headline: Three Boobs Are Most Likely Two Boobs and a Lie


In [7]:
transformer = SentenceTransformer('google/electra-small-discriminator')
text_embeddings_body = transformer.encode(raw_texts)
text_embeddings_headline = transformer.encode(raw_headlines)
text_embeddings = text_embeddings_body + text_embeddings_headline

No sentence-transformers model found with name google/electra-small-discriminator. Creating a new one with mean pooling.


In [8]:
model = LogisticRegression(max_iter=400)

pred_probs = cross_val_predict(model, text_embeddings, labels, method="predict_proba")

In [9]:
data_dict = {"texts": raw_texts, "labels": labels}

In [10]:
lab = Datalab(data_dict, label_name="labels")
lab.find_issues(pred_probs=pred_probs, features=text_embeddings)

Finding null issues ...
Finding label issues ...
Finding outlier issues ...
Finding near_duplicate issues ...
Finding non_iid issues ...
Finding class_imbalance issues ...
Finding underperforming_group issues ...

Audit complete. 1845 issues found in the dataset.


In [11]:
lab.report()

Dataset Information: num_examples: 4000, num_classes: 4

Here is a summary of various issues found in your data:

    issue_type  num_issues
         label        1697
       outlier          75
near_duplicate          72
       non_iid           1

Learn about each issue: https://docs.cleanlab.ai/stable/cleanlab/datalab/guide/issue_type_description.html
See which examples in your dataset exhibit each issue via: `datalab.get_issues(<ISSUE_NAME>)`

Data indices corresponding to top examples of each issue are shown below.


----------------------- label issues -----------------------

About this issue:
	Examples whose given label is estimated to be potentially incorrect
    (e.g. due to annotation error) are flagged as having label issues.
    

Number of examples with this issue: 1697
Overall dataset quality in terms of this issue: 0.5120

Examples representing most severe instances of this issue:
      is_label_issue  label_score given_label predicted_label
2775            True     0.0

In [12]:
label_issues = lab.get_issues("label")
label_issues.head()

,is_label_issue,label_score,given_label,predicted_label
0,False,0.601746,agree,agree
1,True,0.010235,agree,disagree
2,False,0.568817,agree,agree
3,True,0.241516,agree,disagree
4,True,0.137173,agree,disagree


In [13]:
identified_label_issues = label_issues[label_issues["is_label_issue"] == True]
lowest_quality_labels = label_issues["label_score"].argsort()[:5].to_numpy()

print(
    f"cleanlab found {len(identified_label_issues)} potential label errors in the dataset.\n"
    f"Here are indices of the top 5 most likely errors: \n {lowest_quality_labels}"
)

cleanlab found 1697 potential label errors in the dataset.
Here are indices of the top 5 most likely errors: 
 [2775 2241 1496  644  574]


In [15]:
data_with_suggested_labels = pd.DataFrame(
    {"text": raw_texts, "headlines": raw_headlines, "given_label": labels, "suggested_label": label_issues["predicted_label"]}
)
data_with_suggested_labels.iloc[lowest_quality_labels]

,text,headlines,given_label,suggested_label
2775,Eggs are the Dr. Jekylll and Mr. Hyde of the b...,Eggs are nearly as bad for your heart as cigar...,discuss,disagree
2241,In this video posted to YouTube and all over R...,Son Pays Off His Parents’ Mortgage for Christmas,discuss,agree
1496,An Iraqi airstrike on Thursday (September 4th)...,Islamic State Leader al-Baghdadi “Not Dead”,disagree,discuss
644,Nicaraguan officials say a meteorite gouged ou...,Doubts cast over Nicaragua meteorite claim,agree,disagree
574,Six-time MLB All-Star Jose Canseco shot himsel...,Jose Canseco Accidentally Shot Himself in the ...,agree,discuss
